In [1]:
!pip install -U sentence-transformers

  Using cached torch-2.10.0-cp314-cp314-macosx_14_0_arm64.whl.metadata (31 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 5.6 MB/s  0:00:01m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 7.3 MB/s  0:00:00 eta 0:00:01
Using cached torch-2.10.0-cp314-cp314-macosx_14_0_arm64.whl (79.5 MB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp314-cp314-macosx_11_0_arm64.whl 

In [ ]:
# Load RecruitView dataset and parse transcripts by timestamp
import re
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("AI4A-lab/RecruitView")
train = dataset["train"]

def parse_transcript(transcript_str):
    """Parse '[00:01 - 00:11] text' lines into list of (timestamp, text)."""
    if not transcript_str or not transcript_str.strip():
        return []
    segments = []
    for line in transcript_str.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("[") and "]" in line:
            idx = line.index("]")
            timestamp = line[1:idx].strip()
            text = line[idx + 1 :].strip()
            if text:
                segments.append((timestamp, text))
        else:
            segments.append(("", line))
    return segments

# Use personality_score from dataset if present, else placeholder
personality_col = None
for col in ("personality_score", "overall_personality", "personality"):
    if col in train.column_names:
        personality_col = col
        break

# Build table: participant_id, timestamp, transcript, personality_score
rows = []
for participant_id in range(len(train)):
    transcript_str = train["transcript"][participant_id]
    score = train[personality_col][participant_id] if personality_col else None
    for timestamp, text in parse_transcript(transcript_str):
        rows.append({
            "participant_id": participant_id,
            "timestamp": timestamp,
            "transcript": text,
            "personality_score": score,
        })

table = pd.DataFrame(rows)
print(f"Loaded {len(train)} participants → {len(table)} segments.")
print("Table columns:", list(table.columns))
print(table.head())

In [ ]:
# Embed segment transcripts and add transcript_embeddings to table
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
segment_texts = table["transcript"].tolist()
embeddings = model.encode(segment_texts, show_progress_bar=True)

# Add transcript_embeddings column to table (store as list per row for DataFrame)
table["transcript_embeddings"] = [embeddings[i] for i in range(len(embeddings))]

print(f"Embeddings shape: {embeddings.shape}")
print("Table columns:", table.columns.tolist())
table.head()